# PlantAdvisor — LLM-as-judge

Google Colab, GPU T4. Оценка сохранённых ответов; модель 3B повторно не запускается.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/Batyaoftheyear/PlantAdvisor"
REPO_REF = "main"
PROJECT = Path("/content/PlantAdvisorJudge")
SOURCE_DIR = "results/baseline"
OUTPUT_ROOT = "outputs"
REFERENCE_RUN = None
RUN_FULL_JUDGE = False
DOWNLOAD_RESULTS = False


## 1. Код и зависимости


In [ ]:
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)
if not Path("src/plant_advisor/judge.py").is_file():
    raise RuntimeError("В выбранной копии нет кода судьи. Проверьте ветку репозитория.")
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Выберите GPU T4 в настройках Colab")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-judge.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation", "-e", "."], check=True)


## 2. Проверка baseline и кода


In [ ]:
subprocess.run([sys.executable, "-m", "plant_advisor", "audit"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"], check=True)
RESULT_FILE = Path(OUTPUT_ROOT) / "judge_last_run.json"
judge_args = [sys.executable, "-m", "plant_advisor", "judge",
              "--source-dir", SOURCE_DIR, "--output-root", OUTPUT_ROOT,
              "--result-file", str(RESULT_FILE)]
if REFERENCE_RUN is not None:
    judge_args += ["--reference-run", REFERENCE_RUN]


## 3. Пробная оценка: q01, q02, q26

Проверьте объяснения. В q02 при слишком влажном воздухе предлагается повысить влажность; судья должен заметить эту ошибку.


In [ ]:
subprocess.run(judge_args, check=True)
result = json.loads(RESULT_FILE.read_text(encoding="utf-8"))
RUN_DIR = Path(result["run_dir"])
import pandas as pd
from IPython.display import display

sanity = pd.read_csv(RUN_DIR / "sanity/scores.csv")
with pd.option_context("display.max_colwidth", None):
    display(sanity[["id", "task", "status", "factual_correctness", "factual_correctness_explanation",
                    "actionability", "actionability_explanation", "diagnostic_reasoning",
                    "diagnostic_reasoning_explanation", "critical_error", "critical_error_explanation"]])
print("Все три JSON-оценки валидны:", result["complete"])
if result["complete"]:
    print("Известная ошибка повлияла на оценку:", result.get("known_error_penalized", "не проверяется для этого источника"))
else:
    print("Проверка известной ошибки не завершена: есть невалидные ответы судьи")


### Диагностика JSON

При `invalid_json` выполните эту ячейку и сохраните её вывод или скачайте архив. Модель повторно не запускается.


In [ ]:
from pathlib import Path
import json

if "RUN_DIR" not in globals():
    result_file = Path("outputs/judge_last_run.json")
    RUN_DIR = Path(json.loads(result_file.read_text(encoding="utf-8"))["run_dir"])
records = [json.loads(line) for line in (RUN_DIR / "sanity/judgments.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
invalid = [row for row in records if row["status"] != "ok"]
for row in invalid:
    print("\nВопрос:", row["id"], "Статус:", row["status"])
    for number, attempt in enumerate(row["attempts"], 1):
        print("Попытка:", number, "Ошибка:", attempt.get("parse_error"))
        print("Завершение:", attempt.get("finish_reason"), "Токенов:", attempt.get("output_tokens"))
        print(attempt["raw_output"])
if not invalid:
    print("Невалидных JSON нет")


## 4. Оценить все 30 ответов

После проверки трёх объяснений задайте `RUN_FULL_JUDGE=True` в первой ячейке и повторите её и эту ячейку.


In [ ]:
if RUN_FULL_JUDGE:
    subprocess.run(judge_args + ["--full"], check=True)
    result = json.loads(RESULT_FILE.read_text(encoding="utf-8"))
    RUN_DIR = Path(result["run_dir"])
    display(pd.read_csv(RUN_DIR / "full/scores.csv"))
    display(pd.read_csv(RUN_DIR / "full/metrics.csv"))
    print("Покрытие валидными оценками:", result["coverage"])
    print("Все 30 оценок корректны:", result["complete"])
else:
    print("Полный запуск отключён: RUN_FULL_JUDGE=False")


## 5. Скачать оценки и настройки

Задайте `DOWNLOAD_RESULTS=True` и выполните ячейку до завершения сессии Colab.


In [ ]:
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files
    archive = shutil.make_archive("/content/plant_advisor_judge", "zip", root_dir=RUN_DIR)
    files.download(archive)
else:
    print("Скачивание отключено: DOWNLOAD_RESULTS=False")
